# Notebook 06 - RAG Pipeline

This notebook runs the RAG pipeline end to end. It is a smoke test — the goal is
to confirm the pipeline works, not to measure quality. The real evaluation
happens later, on the 150-question human set.

The pipeline does four things:

1. Embeds the query with `BAAI/bge-m3`.
2. Searches a FAISS `IndexFlatIP` index that holds 727,138 chunk vectors.
3. Reranks the candidates with `cross-encoder/ms-marco-MiniLM-L-12-v2` and keeps the best few.
4. Sends the evidence to `openai/gpt-oss-120b` on Groq and appends a medical disclaimer.

Two things are checked here:

- **Retrieval** — do the top chunks look relevant to the query?
- **Fallback behaviour** — when the LLM decides the retrieved evidence is not
  good enough, it answers from its own knowledge. We want to know how often
  that happens and why.

Building the pipeline takes a few minutes on the first run, because it loads the
embedding model, the 3 GB FAISS index, the 511 MB chunk mapping and the reranker.
The cell after the imports will look stuck for a while; that is normal.


## 1. Imports and path setup

The notebook sits in `notebooks/`, so the repo root is one level up. We add the
repo root to `sys.path` so `src.*` and `config.*` import, and load `.env` so the
Groq keys are available.


In [1]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np

# notebook lives in notebooks/, so the repo root is one level up
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

print("Repo root:", ROOT)
print("Python:", sys.version.split()[0])


Repo root: d:\Projects\Healthcare-RAG-Assistant
Python: 3.12.10


## 2. Build the pipeline

`build_rag_pipeline()` returns a cached `RAGPipeline`. It loads the embedding
model, the FAISS index, the chunk mapping, the optional category classifier and
the cross-encoder reranker, so this cell needs a few minutes on a cold start.

Facts about the current knowledge base:

- Index file: `data/embeddings/faiss_index/kb_index_flatip.faiss` — 727,138 vectors, `IndexFlatIP`, 1024 dimensions.
- Mapping file: `data/embeddings/faiss_index/kb_chunk_mapping.pkl` with the columns `chunk_id`, `doc_id`, `source`, `source_id`, `category`, `text_chunk`. There are no `question`, `context` or `answer` columns any more; the pipeline derives those from `text_chunk`.
- Embedding model: `BAAI/bge-m3`. Reranker: `cross-encoder/ms-marco-MiniLM-L-12-v2`.
- Settings in `config/settings.py`: `TOP_K=30`, `INJECT_K=5`, `MAX_CONTEXT_WORDS=250`, `USE_RERANKER=True`.
- Generation runs on Groq with `openai/gpt-oss-120b`. `.env` holds 31 comma-separated `GROQ_API_KEY` values, and `src/rag/pipeline.py` loads `.env` at import time, so no extra setup is needed before `build_rag_pipeline()`.
- Retrieval is dense embedding search plus reranking only. No keyword index is built.


In [2]:
from src.rag.pipeline import build_rag_pipeline

start = time.time()
pipeline = build_rag_pipeline()
print(f"Pipeline built in {time.time() - start:.1f}s")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

📥 Local model not found. Downloading from HuggingFace: AbdoMatrix/biobert-medical-classifier


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Classifier loaded | Device: cuda | Classes: ['Diagnosis', 'General', 'Medication', 'Prevention', 'Symptoms', 'Treatment']


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Pipeline built in 25.7s


## 3. What got loaded

Read the numbers straight off the pipeline object instead of trusting the docs.


In [3]:
mapping = pipeline.mapping_df

print("Index type:", type(pipeline.index).__name__)
print("Index vectors:", f"{pipeline.index.ntotal:,}")
print("Index dimensions:", pipeline.index.d)
print("Mapping rows:", f"{len(mapping):,}")
print("Mapping columns:", mapping.columns.tolist())
print("Embedding model:", pipeline.encoder[0].auto_model.config.name_or_path)
print("top_k (candidates fetched):", pipeline.top_k)
print("inject_k (chunks sent to the LLM):", pipeline.inject_k)
print("max_context_words:", pipeline.max_context_words)
print("max_new_tokens:", pipeline.max_new_tokens)
print("Reranker enabled:", pipeline._use_reranker)
print("Reranker model:", pipeline.reranker.config._name_or_path if pipeline._use_reranker else "not loaded")
print("Groq client active:", pipeline._use_groq)
print("Groq model:", getattr(pipeline, "_groq_model", "n/a"))
print("Groq keys loaded:", len(getattr(pipeline, "_groq_clients", [])))
print("Classifier active:", pipeline._use_classifier)

print()
first = mapping.iloc[0]
print("First mapping row:")
print("  chunk_id:", first["chunk_id"], "| doc_id:", first["doc_id"], "| source:", first["source"],
      "| source_id:", first["source_id"], "| category:", first["category"])
print("  text_chunk (first 200 chars):", " ".join(str(first["text_chunk"]).split())[:200])


Index type: IndexFlatIP
Index vectors: 727,138
Index dimensions: 1024
Mapping rows: 727,138
Mapping columns: ['chunk_id', 'doc_id', 'source', 'source_id', 'category', 'text_chunk']
Embedding model: BAAI/bge-m3
top_k (candidates fetched): 30
inject_k (chunks sent to the LLM): 5
max_context_words: 250
max_new_tokens: 2048
Reranker enabled: True
Reranker model: cross-encoder/ms-marco-MiniLM-L-12-v2
Groq client active: True
Groq model: openai/gpt-oss-120b
Groq keys loaded: 31
Classifier active: True

First mapping row:
  chunk_id: 0 | doc_id: 0 | source: medquad | source_id: 3005 | category: treatment
  text_chunk (first 200 chars): Question: What are the treatments for Breast Cancer? Answer: Key Points - There are different types of treatment for patients with breast cancer. - Five types of standard treatment are used: - Surgery


## 4. Retrieval test

Five queries, one per category the classifier knows. Each `retrieve` call fetches
`top_k` candidates, reranks them and returns the best five. We print the chunk id,
the source, the reranker score and a short excerpt.

This is a retrieval-only check — no LLM is called. It runs in seconds.


In [4]:
category_queries = {
    "Symptoms":   "What are the symptoms of type 2 diabetes?",
    "Diagnosis":  "How is pneumonia diagnosed?",
    "Treatment":  "What are the treatment options for hypertension?",
    "Medication": "What are the side effects of metformin?",
    "Prevention": "How can cardiovascular disease be prevented?",
}

for category, query in category_queries.items():
    chunks = pipeline.retrieve(query, top_k=5)
    print("=" * 95)
    print(f"{category}: {query}")
    for rank, chunk in enumerate(chunks, 1):
        excerpt = " ".join(chunk["text_chunk"].split())[:130]
        print(f"  {rank}. chunk_id={chunk['chunk_id']} source={chunk['source']} "
              f"reranker_score={chunk['reranker_score']:.3f} distance={chunk['distance']:.3f}")
        print(f"     {excerpt}")
    print()


Symptoms: What are the symptoms of type 2 diabetes?
  1. chunk_id=17938 source=medquad reranker_score=9.252 distance=0.777
     Question: What is (are) Diabetes Type 2? . The symptoms of type 2 diabetes appear slowly. Some people do not notice symptoms at al
  2. chunk_id=23076 source=medquad reranker_score=9.243 distance=0.777
     Question: What are the symptoms of Am I at Risk for Type 2 Diabetes? Taking Steps to Lower Your Risk of Getting Diabetes? Answer: 
  3. chunk_id=4782 source=medquad reranker_score=8.157 distance=0.771
     Question: What are the symptoms of Diabetes? Answer: Diabetes is often called a "silent" disease because it can cause serious comp
  4. chunk_id=25940 source=medquad reranker_score=7.945 distance=0.765
     Question: What are the symptoms of Your Guide to Diabetes: Type 1 and Type 2? Answer: The signs and symptoms of diabetes are - bei
  5. chunk_id=29975 source=medlineplus reranker_score=6.271 distance=0.770
     Question: What is Diabetes Type 2? . Have

## 5. Full pipeline on one query

`pipeline.answer(query)` runs the whole chain: routing guard, retrieval,
reranking, generation and the disclaimer. It returns a dict, and we print the
fields we care about.


In [5]:
result = pipeline.answer("What are the symptoms of diabetes?")

print("Question:", result["question"])
print()
print("Answer:")
print(result["answer"])
print()
print("answer_source:", result["answer_source"])
print("used_rag:", result["used_rag"])
print("disclaimer_present:", result["disclaimer_present"])
print("sources returned:", len(result["retrieved_sources"]))
print("top_k:", result["top_k"])
print("retrieval_quality (mean reranker score):", result["retrieval_quality"])
print("mean_cosine_similarity:", result["mean_cosine_similarity"])
print()
print("First three sources:")
for source in result["retrieved_sources"][:3]:
    print(f"  chunk_id={source['chunk_id']} category={source['category']} "
          f"reranker_score={source['reranker_score']} cosine={source['distance']}")


Question: What are the symptoms of diabetes?

Answer:
Diabetes often shows up with several common signs, although some people may have no noticeable symptoms at all. • Feeling very thirsty (excessive thirst) • Urinating a lot, especially at night (frequent urination) • Feeling very hungry even after eating (increased appetite) • Feeling unusually tired or weak (fatigue) • Losing weight without trying (unexplained weight loss) • Blurry or changed vision (difficulty seeing clearly) • Dry, itchy skin that may heal slowly (slow‑healing sores) • Numbness, tingling, or loss of feeling in the feet or hands (nerve irritation) In type 1 diabetes these symptoms usually appear quickly, while in type 2 they often develop slowly and may be mild enough to go unnoticed; gestational diabetes often has no symptoms at all. If you notice any of these signs, a doctor can confirm diabetes with a simple blood test.

⚠️ MEDICAL DISCLAIMER: This response is generated by an AI system for educational and inform

## 6. Ten-query logged run

Ten queries across the categories the classifier knows. All of them target
content that the knowledge base should cover — consumer questions from MedQuAD
and MedlinePlus, plus research-style questions from PubMedQA.

Queries that are clearly outside the knowledge base (alternative medicine,
drug labels, edge-case clinical decisions) are left for the evaluation set,
not for this smoke test.

For each query we print the latency, where the answer came from, and the first
200 characters of the answer. The full dicts are kept in `results` for the
summary, the fallback analysis, and the log file.


In [6]:
test_queries = [
    # Symptoms
    "What are the symptoms of type 2 diabetes?",
    "What are the common symptoms of asthma?",
    # Diagnosis
    "How is pneumonia diagnosed?",
    "How is breast cancer diagnosed?",
    # Treatment
    "What are the treatment options for hypertension?",
    "How is type 2 diabetes treated?",
    # Prevention
    "How can cardiovascular disease be prevented?",
    "How can type 2 diabetes be prevented?",
    # General
    "What is the immune system?",
    "What is Alzheimer's disease?",
]

print(f"{len(test_queries)} test queries ready")
for number, query in enumerate(test_queries, 1):
    print(f"  {number}. {query}")


10 test queries ready
  1. What are the symptoms of type 2 diabetes?
  2. What are the common symptoms of asthma?
  3. How is pneumonia diagnosed?
  4. How is breast cancer diagnosed?
  5. What are the treatment options for hypertension?
  6. How is type 2 diabetes treated?
  7. How can cardiovascular disease be prevented?
  8. How can type 2 diabetes be prevented?
  9. What is the immune system?
  10. What is Alzheimer's disease?


In [7]:
results = []
latencies = []


def json_safe_sources(sources):
    """Some rows in the mapping have no category (pandas NA), which json cannot write,
    so each source is rebuilt with plain Python types before it goes into the log."""
    records = []
    for source in sources:
        category = source["category"] if isinstance(source["category"], str) else ""
        records.append({
            "chunk_id": int(source["chunk_id"]),
            "question": str(source["question"]),
            "category": category,
            "category_score": float(source["category_score"]),
            "distance": float(source["distance"]),
            "relevance_score": float(source["relevance_score"]),
            "reranker_score": float(source["reranker_score"]),
            "context": str(source["context"]),
            "answer": str(source["answer"]),
            "excerpt": str(source["excerpt"]),
        })
    return records


print("=" * 95)
print("10-QUERY LOGGED RUN")
print("=" * 95)

for number, query in enumerate(test_queries, 1):
    start = time.time()
    result = pipeline.answer(query)
    elapsed_ms = (time.time() - start) * 1000
    latencies.append(elapsed_ms)

    results.append({
        "query_num": number,
        "question": result["question"],
        "answer_raw": result["answer_raw"],
        "disclaimer_present": result["disclaimer_present"],
        "top_k": result["top_k"],
        "used_rag": result["used_rag"],
        "answer_source": result["answer_source"],
        "retrieval_quality": result["retrieval_quality"],
        "mean_cosine_similarity": result["mean_cosine_similarity"],
        "sources": json_safe_sources(result["retrieved_sources"]),
        "latency_ms": round(elapsed_ms, 2),
    })

    print()
    print("-" * 95)
    print(f"Query {number}: {query}")
    print(f"Latency: {elapsed_ms:.0f} ms")
    print(f"Sources: {result['top_k']} | answer_source: {result['answer_source']} | used_rag: {result['used_rag']}")
    print(f"Source chunk ids: {[s['chunk_id'] for s in result['retrieved_sources']]}")
    print(f"Answer (first 200 chars): {result['answer_raw'][:200]}")


10-QUERY LOGGED RUN

-----------------------------------------------------------------------------------------------
Query 1: What are the symptoms of type 2 diabetes?
Latency: 2827 ms
Sources: 30 | answer_source: rag | used_rag: True
Source chunk ids: [17938, 23076, 4782, 26667, 25940, 3483, 10134, 29975, 29948, 17937, 29973, 4786, 29971, 189088, 529784, 5886, 10008, 7982, 10968, 4783, 7022, 16352, 11037, 26763, 670885, 20528, 4785, 10969, 5888, 9999]
Answer (first 200 chars): Type 2 diabetes often develops slowly and many people may not notice any signs at first. Common symptoms include: • feeling very thirsty • urinating often, especially at night • feeling unusually hung

-----------------------------------------------------------------------------------------------
Query 2: What are the common symptoms of asthma?
Latency: 6331 ms
Sources: 30 | answer_source: rag | used_rag: True
Source chunk ids: [8715, 29356, 29362, 29360, 21281, 716553, 11295, 628423, 29353, 29363, 681689, 66031

## 7. Fallback investigation

When the LLM decides the retrieved evidence is not good enough, it answers from
its own knowledge (`answer_source = general_knowledge`). This section looks at
which queries fell back, and whether the decision matches the retrieval quality.

Two possible patterns:

- **Fallbacks have clearly lower reranker scores.** The LLM is being reasonable — it saw weak evidence and gave up. Nothing to fix in the pipeline, but it confirms the KB does not cover those topics.
- **Fallbacks have similar scores to the RAG answers.** The LLM is ignoring usable evidence. That points at the prompt, not at retrieval.


In [8]:
rag_results = [r for r in results if r["answer_source"] == "rag"]
fallback_results = [r for r in results if r["answer_source"] == "general_knowledge"]

print(f"RAG answers:      {len(rag_results)}")
print(f"Fallback answers: {len(fallback_results)}")
print()

if rag_results:
    print(f"Mean retrieval_quality for RAG answers:      "
          f"{np.mean([r['retrieval_quality'] for r in rag_results]):.3f}")
if fallback_results:
    print(f"Mean retrieval_quality for fallback answers: "
          f"{np.mean([r['retrieval_quality'] for r in fallback_results]):.3f}")
print()

for r in fallback_results:
    print("=" * 90)
    print(f"Query {r['query_num']}: {r['question']}")
    print(f"  mean reranker score:   {r['retrieval_quality']:.3f}")
    print(f"  mean cosine similarity: {r['mean_cosine_similarity']:.3f}")
    print(f"  top 3 retrieved chunks:")
    for s in r["sources"][:3]:
        excerpt = " ".join(s["excerpt"].split())[:140]
        print(f"    [{s['chunk_id']}] rerank={s['reranker_score']:.2f} "
              f"cosine={s['distance']:.3f} category={s['category'] or 'n/a'}")
        print(f"      {excerpt}")
    print()

if not fallback_results:
    print("All queries were answered from the retrieved evidence.")


RAG answers:      10
Fallback answers: 0

Mean retrieval_quality for RAG answers:      1.533

All queries were answered from the retrieved evidence.


## 8. Summary and log

Counts, latency stats and the ratio of RAG-sourced to fallback answers. Then the
run is saved to `reports/rag_pipeline_test_log.json`.

Note on `mean retrieval_quality`: the pipeline returns 30 sources per query
(`TOP_K=30`), and this metric is the average of the reranker scores across all
30. The tail of those 30 is weak, so the average sits near zero even when the
top 5 are strong. Read it as a shape indicator, not as a quality score.


In [9]:
mean_latency = sum(latencies) / len(latencies)
rag_answers = sum(1 for r in results if r["answer_source"] == "rag")
general_answers = sum(1 for r in results if r["answer_source"] == "general_knowledge")

print("=" * 60)
print("10-QUERY TEST SUMMARY")
print("=" * 60)
print(f"Queries tested:                 {len(results)}")
print(f"Disclaimer in every answer:     {all(r['disclaimer_present'] for r in results)}")
print(f"Sources retrieved per query:    {[r['top_k'] for r in results]}")
print(f"Answers from retrieved evidence: {rag_answers}")
print(f"Answers from general knowledge:  {general_answers}")
print()
print("Latency (ms):")
print(f"  min:   {min(latencies):.0f}")
print(f"  max:   {max(latencies):.0f}")
print(f"  mean:  {mean_latency:.0f}")
print(f"  total: {sum(latencies) / 1000:.1f} s")
print()
print("Mean retrieval quality (reranker score over all 30 sources): "
      f"{sum(r['retrieval_quality'] for r in results) / len(results):.3f}")
print("Mean cosine similarity of retrieved chunks: "
      f"{sum(r['mean_cosine_similarity'] for r in results) / len(results):.3f}")

# Hard checks. If either fails, the pipeline is not in a runnable state
# for a demo, even if individual answers still look reasonable.
for r in results:
    assert r["disclaimer_present"], f"missing disclaimer for: {r['question']}"

assert rag_answers >= 7, (
    f"Too many fallbacks: only {rag_answers}/10 came from the retrieved evidence "
    f"({general_answers} fell back to general knowledge). "
    f"Check the fallback section above before continuing."
)

print()
print(f"Checks passed: every answer carried a disclaimer, "
      f"and {rag_answers}/10 answers used retrieved evidence.")


10-QUERY TEST SUMMARY
Queries tested:                 10
Disclaimer in every answer:     True
Sources retrieved per query:    [30, 30, 30, 30, 30, 30, 30, 30, 30, 30]
Answers from retrieved evidence: 10
Answers from general knowledge:  0

Latency (ms):
  min:   2827
  max:   26201
  mean:  18799
  total: 188.0 s

Mean retrieval quality (reranker score over all 30 sources): 1.533
Mean cosine similarity of retrieved chunks: 0.678

Checks passed: every answer carried a disclaimer, and 10/10 answers used retrieved evidence.


In [10]:
log_path = ROOT / "reports" / "rag_pipeline_test_log.json"
tmp_path = log_path.with_suffix(".json.tmp")

# write next to the target and rename, so an interrupted run cannot leave a half-written log
with open(tmp_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
os.replace(tmp_path, log_path)

print("Test log saved to:", log_path)
print("Queries in log:", len(results))
print("Sources per query:", [len(r["sources"]) for r in results])
print("Answer sources:", [r["answer_source"] for r in results])


Test log saved to: d:\Projects\Healthcare-RAG-Assistant\reports\rag_pipeline_test_log.json
Queries in log: 10
Sources per query: [30, 30, 30, 30, 30, 30, 30, 30, 30, 30]
Answer sources: ['rag', 'rag', 'rag', 'rag', 'rag', 'rag', 'rag', 'rag', 'rag', 'rag']


## Summary

| Item | Value |
|---|---|
| Embedding model | `BAAI/bge-m3` (1024-dim) |
| Vector store | FAISS `IndexFlatIP`, 727,138 vectors |
| Reranker | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| Retrieval depth | `TOP_K=30` candidates, `INJECT_K=5` chunks to the LLM |
| Generator | `openai/gpt-oss-120b` on Groq |
| Disclaimer | Appended to every answer |
| Smoke test | 10 in-scope queries across the 5 categories |
| Fallback check | Queries that fell back to general knowledge, with their top chunks |
| Hard checks | Every answer carries a disclaimer; ≥ 7/10 answers use retrieved evidence |
| Test log | `reports/rag_pipeline_test_log.json` |

Next notebook: `07_classification_model.ipynb` for the category classifier.
